In [ ]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from torch.utils.data import Dataset, DataLoader
from models_related import BP_Model, evaluate_torch_model, BP_Model_DO

# Load your data from the .npy file
data = np.load('C:\\Users\\user\\Documents\\Python Scripts\\mimic4_abp_hybridnn\\data\\final_records_mimic4.npy', allow_pickle=True)

# Initialize lists to hold the data
ecg_segments = []
ppg_segments = []
sbp_values = []
dbp_values = []

for record in data:
    ecg_segments.append(record['ecg'])
    ppg_segments.append(record['ppg'])
    sbp_values.append(record['true_sbp'])
    dbp_values.append(record['true_dbp'])

# Convert to NumPy arrays
ecg_segments = np.array(ecg_segments)
ppg_segments = np.array(ppg_segments)
sbp_values = np.array(sbp_values)
dbp_values = np.array(dbp_values)

# Stack ECG and PPG signals along the last axis
combined_segments = np.stack((ecg_segments, ppg_segments), axis=-1)  # Shape: (num_samples, segment_length, 2)
combined_segments = combined_segments.astype(np.float32)
sbp_values = sbp_values.astype(np.float32)
dbp_values = dbp_values.astype(np.float32)

# BP Dataset class
class BP_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y.unsqueeze(1)  # Ensure y is of shape (N, 1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Load your saved models for SBP and DBP
model_sbp = BP_Model()  # Replace with your model definition
model_dbp = BP_Model()  # Replace with your model definition

# Load saved model weights
model_sbp.load_state_dict(torch.load('C:\\Users\\user\\Documents\\Python Scripts\\mimic4_abp_hybridnn\\model\\torch_cnn_lstm_sbp.pth'))  
model_dbp.load_state_dict(torch.load('C:\\Users\\user\\Documents\\Python Scripts\\mimic4_abp_hybridnn\\model\\torch_cnn_lstm_dbp.pth'))

# Move models to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sbp.to(device)
model_dbp.to(device)

# Set up KFold cross-validation with 10 splits
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Initialize variables for evaluation
all_predictions = []
all_targets = []

batch_size = 128

# Loop over each fold
for fold, (train_idx, val_idx) in enumerate(kf.split(combined_segments)):
    print(f"Starting fold {fold+1}")
    
    # Split data into training and validation for this fold
    X_train_fold, X_val_fold = combined_segments[train_idx], combined_segments[val_idx]
    y_val_fold_sbp = sbp_values[val_idx]
    y_val_fold_dbp = dbp_values[val_idx]

    # Convert to tensors
    X_val_fold_tensor = torch.tensor(X_val_fold)
    y_val_fold_sbp_tensor = torch.tensor(y_val_fold_sbp)
    y_val_fold_dbp_tensor = torch.tensor(y_val_fold_dbp)

    # Create datasets for SBP and DBP
    val_dataset_sbp = BP_Dataset(X_val_fold_tensor, y_val_fold_sbp_tensor)
    val_dataset_dbp = BP_Dataset(X_val_fold_tensor, y_val_fold_dbp_tensor)

    # DataLoaders for validation sets
    val_loader_sbp = DataLoader(val_dataset_sbp, batch_size=batch_size, shuffle=False)
    val_loader_dbp = DataLoader(val_dataset_dbp, batch_size=batch_size, shuffle=False)

    # Evaluate the model on the validation set using your function
    predictions_sbp, targets_sbp = evaluate_torch_model(model_sbp, val_loader_sbp, device)
    predictions_dbp, targets_dbp = evaluate_torch_model(model_dbp, val_loader_dbp, device)

    # Store predictions and targets for further analysis
    all_predictions.append((predictions_sbp, predictions_dbp))
    all_targets.append((targets_sbp, targets_dbp))

In [1]:
import numpy as np
import torch
from sklearn.model_selection import KFold
from torch.utils.data import Dataset, DataLoader
from models_related import BP_Model, evaluate_torch_model, BP_Model_DO
import pyarrow.parquet as pq

# Load your saved models for SBP and DBP
model_sbp = BP_Model_DO()
model_dbp = BP_Model()

# Load saved model weights
model_sbp.load_state_dict(torch.load('C:\\Users\\user\\Documents\\Python Scripts\\mimic4_abp_hybridnn\\model\\torch_cnn_lstm_sbp.pth'))  
model_dbp.load_state_dict(torch.load('C:\\Users\\user\\Documents\\Python Scripts\\mimic4_abp_hybridnn\\model\\torch_cnn_lstm_dbp.pth'))

# Move models to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sbp.to(device)
model_dbp.to(device)

# Initialize variables for evaluation
all_predictions = []
all_targets = []

batch_size = 128
chunk_size = 5000  # Number of rows to process at once

# Define a Dataset class for blood pressure data
class BP_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y.unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Read Parquet file in chunks
parquet_file = 'data/final_records_mimic3_all_with_id.parquet'
for batch in pq.ParquetFile(parquet_file).iter_batches(batch_size=chunk_size):
    # Convert batch to DataFrame
    data_chunk = batch.to_pandas()
    
    # Extract data from DataFrame
    ecg_segments = np.array(data_chunk['ecg'].tolist())
    ppg_segments = np.array(data_chunk['ppg'].tolist())
    sbp_values = np.array(data_chunk['true_sbp'].tolist(), dtype=np.float32)
    dbp_values = np.array(data_chunk['true_dbp'].tolist(), dtype=np.float32)
    combined_segments = np.stack((ecg_segments, ppg_segments), axis=-1).astype(np.float32)

    # Perform K-Fold cross-validation on each chunk independently
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    for fold, (train_idx, val_idx) in enumerate(kf.split(combined_segments)):
        print(f"Processing fold {fold + 1} in chunk")

        # Split data for this fold
        X_train_fold, X_val_fold = combined_segments[train_idx], combined_segments[val_idx]
        y_val_fold_sbp = sbp_values[val_idx]
        y_val_fold_dbp = dbp_values[val_idx]

        # Convert validation data to tensors
        X_val_fold_tensor = torch.tensor(X_val_fold)
        y_val_fold_sbp_tensor = torch.tensor(y_val_fold_sbp)
        y_val_fold_dbp_tensor = torch.tensor(y_val_fold_dbp)

        # Create datasets and dataloaders for SBP and DBP
        val_dataset_sbp = BP_Dataset(X_val_fold_tensor, y_val_fold_sbp_tensor)
        val_dataset_dbp = BP_Dataset(X_val_fold_tensor, y_val_fold_dbp_tensor)
        val_loader_sbp = DataLoader(val_dataset_sbp, batch_size=batch_size, shuffle=False)
        val_loader_dbp = DataLoader(val_dataset_dbp, batch_size=batch_size, shuffle=False)

        # Evaluate the models on the validation sets
        predictions_sbp, targets_sbp = evaluate_torch_model(model_sbp, val_loader_sbp, device)
        predictions_dbp, targets_dbp = evaluate_torch_model(model_dbp, val_loader_dbp, device)

        # Append results for further analysis
        all_predictions.append((predictions_sbp, predictions_dbp))
        all_targets.append((targets_sbp, targets_dbp))


Processing fold 1 in chunk
Processing fold 2 in chunk
Processing fold 3 in chunk
Processing fold 4 in chunk
Processing fold 5 in chunk
Processing fold 6 in chunk
Processing fold 7 in chunk
Processing fold 8 in chunk
Processing fold 9 in chunk
Processing fold 10 in chunk
Processing fold 1 in chunk
Processing fold 2 in chunk
Processing fold 3 in chunk
Processing fold 4 in chunk
Processing fold 5 in chunk
Processing fold 6 in chunk
Processing fold 7 in chunk
Processing fold 8 in chunk
Processing fold 9 in chunk
Processing fold 10 in chunk
Processing fold 1 in chunk
Processing fold 2 in chunk
Processing fold 3 in chunk
Processing fold 4 in chunk
Processing fold 5 in chunk
Processing fold 6 in chunk
Processing fold 7 in chunk
Processing fold 8 in chunk
Processing fold 9 in chunk
Processing fold 10 in chunk
Processing fold 1 in chunk
Processing fold 2 in chunk
Processing fold 3 in chunk
Processing fold 4 in chunk
Processing fold 5 in chunk
Processing fold 6 in chunk
Processing fold 7 in chun

In [9]:
# Calculate final metrics after all folds
from evaluation import evaluate_model_performance
avg_mae_sbp, avg_std_sbp, avg_bhs_5_sbp, avg_bhs_10_sbp, avg_bhs_15_sbp = [], [], [], [], []
avg_mae_dbp, avg_std_dbp, avg_bhs_5_dbp, avg_bhs_10_dbp, avg_bhs_15_dbp = [], [], [], [], []

for fold in range(10):
    # Evaluate SBP metrics
    mae_sbp, rmse_sbp, std_dev_sbp, r2_sbp, bhs_results_5_sbp, bhs_results_10_sbp, bhs_results_15_sbp, _ = evaluate_model_performance(all_targets[fold][0], all_predictions[fold][0])
    avg_mae_sbp.append(mae_sbp)
    avg_std_sbp.append(std_dev_sbp)
    avg_bhs_5_sbp.append(bhs_results_5_sbp)
    avg_bhs_10_sbp.append(bhs_results_10_sbp)
    avg_bhs_15_sbp.append(bhs_results_15_sbp)

    # Evaluate DBP metrics
    mae_dbp, rmse_dbp, std_dev_dbp, r2_dbp, bhs_results_5_dbp, bhs_results_10_dbp, bhs_results_15_dbp, _ = evaluate_model_performance(all_targets[fold][1], all_predictions[fold][1])
    avg_mae_dbp.append(mae_dbp)
    avg_std_dbp.append(std_dev_dbp)
    avg_bhs_5_dbp.append(bhs_results_5_dbp)
    avg_bhs_10_dbp.append(bhs_results_10_dbp)
    avg_bhs_15_dbp.append(bhs_results_15_dbp)

print(f"Average MAE for SBP over 10 folds: {np.mean(avg_mae_sbp)}")
print(f"Average Standard Deviation for SBP over 10 folds: {np.mean(avg_std_sbp)}")
print(f"Average BHS results for SBP (5 mmHg) over 10 folds: {np.mean(avg_bhs_5_sbp)}")
print(f"Average BHS results for SBP (10 mmHg) over 10 folds: {np.mean(avg_bhs_10_sbp)}")
print(f"Average BHS results for SBP (15 mmHg) over 10 folds: {np.mean(avg_bhs_15_sbp)}")

print(f"Average MAE for DBP over 10 folds: {np.mean(avg_mae_dbp)}")
print(f"Average Standard Deviation for DBP over 10 folds: {np.mean(avg_std_dbp)}")
print(f"Average BHS results for DBP (5 mmHg) over 10 folds: {np.mean(avg_bhs_5_dbp)}")
print(f"Average BHS results for DBP (10 mmHg) over 10 folds: {np.mean(avg_bhs_10_dbp)}")
print(f"Average BHS results for DBP (15 mmHg) over 10 folds: {np.mean(avg_bhs_15_dbp)}")


Model Performance Metrics:
MAE: 5.0401
RMSE: 6.2484
Standard Deviation of Errors: 5.8480
R2 Score: -0.1416
BHS Protocol Results:
Within 5 mmHg: 58.40%
Within 10 mmHg: 89.80%
Within 15 mmHg: 98.20%
SBP Model BHS Grade: B
--------------------------------------------------
Model Performance Metrics:
MAE: 2.3809
RMSE: 2.8348
Standard Deviation of Errors: 2.7536
R2 Score: -0.0598
BHS Protocol Results:
Within 5 mmHg: 94.40%
Within 10 mmHg: 99.80%
Within 15 mmHg: 100.00%
SBP Model BHS Grade: A
--------------------------------------------------
Model Performance Metrics:
MAE: 5.0416
RMSE: 6.4914
Standard Deviation of Errors: 6.0659
R2 Score: -0.1452
BHS Protocol Results:
Within 5 mmHg: 62.00%
Within 10 mmHg: 88.20%
Within 15 mmHg: 96.80%
SBP Model BHS Grade: A
--------------------------------------------------
Model Performance Metrics:
MAE: 2.4308
RMSE: 2.8534
Standard Deviation of Errors: 2.8427
R2 Score: -0.0076
BHS Protocol Results:
Within 5 mmHg: 94.40%
Within 10 mmHg: 99.80%
Within 15 mm